In [6]:
%%writefile Chocolaterie_Kengne_app.py
# app.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm
from datetime import datetime, timedelta
import time
import math

# Configuration de la page
st.set_page_config(
    page_title="Chocolaterie Kengne - Risk Management System",
    page_icon="🍫",
    layout="wide",
    initial_sidebar_state="expanded"
)

# CSS personnalisé pour un design professionnel "Dark Finance"
st.markdown("""
<style>
    /* Import police moderne */
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
    
    * {
        font-family: 'Inter', sans-serif;
    }
    
    /* Background principal */
    .stApp {
        background: linear-gradient(135deg, #0a0e1a 0%, #0f1622 50%, #0a0e1a 100%);
    }
    
    /* Sidebar stylisée */
    [data-testid="stSidebar"] {
        background: linear-gradient(180deg, #0d1220 0%, #0a0e1a 100%);
        border-right: 1px solid rgba(46, 204, 113, 0.15);
    }
    
    [data-testid="stSidebar"] .stMarkdown, 
    [data-testid="stSidebar"] .stSelectbox label,
    [data-testid="stSidebar"] .stNumberInput label {
        color: #e0e0e0 !important;
    }
    
    /* Cartes métriques */
    .metric-card {
        background: rgba(18, 24, 36, 0.95);
        border-radius: 16px;
        padding: 1.2rem;
        border: 1px solid rgba(46, 204, 113, 0.2);
        box-shadow: 0 4px 20px rgba(0,0,0,0.3);
        transition: all 0.3s ease;
    }
    
    .metric-card:hover {
        border-color: #2ecc71;
        transform: translateY(-2px);
        box-shadow: 0 8px 30px rgba(46, 204, 113, 0.15);
    }
    
    .metric-label {
        font-size: 0.85rem;
        text-transform: uppercase;
        letter-spacing: 1.5px;
        color: #8892b0;
        font-weight: 500;
    }
    
    .metric-value {
        font-size: 2rem;
        font-weight: 800;
        color: #2ecc71;
        margin-top: 0.5rem;
    }
    
    .metric-sub {
        font-size: 0.75rem;
        color: #5a6a8a;
        margin-top: 0.3rem;
    }
    
    /* Titres */
    h1, h2, h3 {
        font-weight: 700 !important;
        background: linear-gradient(135deg, #ffffff 0%, #8892b0 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }
    
    /* Badges */
    .badge-success {
        background: rgba(46, 204, 113, 0.15);
        color: #2ecc71;
        padding: 0.3rem 0.8rem;
        border-radius: 20px;
        font-size: 0.8rem;
        font-weight: 500;
        border: 1px solid rgba(46, 204, 113, 0.3);
    }
    
    .badge-warning {
        background: rgba(241, 196, 15, 0.15);
        color: #f1c40f;
        padding: 0.3rem 0.8rem;
        border-radius: 20px;
        font-size: 0.8rem;
        font-weight: 500;
        border: 1px solid rgba(241, 196, 15, 0.3);
    }
    
    .badge-danger {
        background: rgba(231, 76, 60, 0.15);
        color: #e74c3c;
        padding: 0.3rem 0.8rem;
        border-radius: 20px;
        font-size: 0.8rem;
        font-weight: 500;
        border: 1px solid rgba(231, 76, 60, 0.3);
    }
    
    /* Boutons */
    .stButton > button {
        background: linear-gradient(135deg, #2ecc71 0%, #27ae60 100%);
        color: white;
        font-weight: 600;
        border: none;
        border-radius: 10px;
        padding: 0.6rem 1.5rem;
        transition: all 0.3s ease;
    }
    
    .stButton > button:hover {
        transform: scale(1.02);
        box-shadow: 0 5px 20px rgba(46, 204, 113, 0.4);
    }
    
    /* Dataframe */
    .dataframe {
        background: rgba(18, 24, 36, 0.9) !important;
        border-radius: 12px !important;
        border: 1px solid rgba(46, 204, 113, 0.2) !important;
    }
    
    /* Animation de pulsation */
    @keyframes pulse {
        0%, 100% { opacity: 1; }
        50% { opacity: 0.5; }
    }
    
    .animate-pulse {
        animation: pulse 2s cubic-bezier(0.4, 0, 0.6, 1) infinite;
    }
    
    /* Footer */
    .footer {
        text-align: center;
        padding: 2rem;
        color: #4a5a7a;
        font-size: 0.75rem;
        border-top: 1px solid rgba(46, 204, 113, 0.1);
        margin-top: 2rem;
    }
</style>
""", unsafe_allow_html=True)

# ============================================================================
# CLASSE PRINCIPALE - Pricing Engine
# ============================================================================

class BlackScholesEngine:
    """Moteur de pricing Black-Scholes avec calcul des Grecs"""
    
    def __init__(self, S0, K, T, r, sigma, option_type='call'):
        self.S0 = S0      # Prix spot
        self.K = K        # Strike
        self.T = T        # Temps (années)
        self.r = r        # Taux sans risque
        self.sigma = sigma # Volatilité
        self.option_type = option_type
    
    def _d1(self):
        return (np.log(self.S0 / self.K) + (self.r + self.sigma**2 / 2) * self.T) / (self.sigma * np.sqrt(self.T))
    
    def _d2(self):
        return self._d1() - self.sigma * np.sqrt(self.T)
    
    def price(self):
        """Prix de l'option"""
        d1 = self._d1()
        d2 = self._d2()
        
        if self.option_type == 'call':
            price = self.S0 * norm.cdf(d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        else:
            price = self.K * np.exp(-self.r * self.T) * norm.cdf(-d2) - self.S0 * norm.cdf(-d1)
        
        return price
    
    def delta(self):
        """Delta : sensibilité au sous-jacent"""
        d1 = self._d1()
        if self.option_type == 'call':
            return norm.cdf(d1)
        else:
            return norm.cdf(d1) - 1
    
    def gamma(self):
        """Gamma : sensibilité du delta"""
        d1 = self._d1()
        return norm.pdf(d1) / (self.S0 * self.sigma * np.sqrt(self.T))
    
    def theta(self):
        """Theta : sensibilité au temps (par jour)"""
        d1 = self._d1()
        d2 = self._d2()
        
        term1 = -(self.S0 * norm.pdf(d1) * self.sigma) / (2 * np.sqrt(self.T))
        
        if self.option_type == 'call':
            term2 = -self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        else:
            term2 = self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(-d2)
        
        return (term1 + term2) / 365  # Theta par jour
    
    def vega(self):
        """Vega : sensibilité à la volatilité"""
        d1 = self._d1()
        return self.S0 * norm.pdf(d1) * np.sqrt(self.T) / 100
    
    def rho(self):
        """Rho : sensibilité au taux d'intérêt"""
        d1 = self._d1()
        d2 = self._d2()
        
        if self.option_type == 'call':
            return self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(d2) / 100
        else:
            return -self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(-d2) / 100
    
    def get_all_greeks(self):
        """Retourne tous les Grecs sous forme de dictionnaire"""
        return {
            'Delta (Δ)': self.delta(),
            'Gamma (Γ)': self.gamma(),
            'Theta (Θ)': self.theta(),
            'Vega (ν)': self.vega(),
            'Rho (ρ)': self.rho()
        }
    
    def implied_volatility(self, market_price, tol=1e-6, max_iter=100):
        """Calcul de la volatilité implicite (Newton-Raphson)"""
        def price_diff(sigma):
            temp_engine = BlackScholesEngine(self.S0, self.K, self.T, self.r, sigma, self.option_type)
            return temp_engine.price() - market_price
        
        sigma_low, sigma_high = 0.01, 2.0
        
        for _ in range(max_iter):
            sigma_mid = (sigma_low + sigma_high) / 2
            diff = price_diff(sigma_mid)
            
            if abs(diff) < tol:
                return sigma_mid
            elif diff < 0:
                sigma_low = sigma_mid
            else:
                sigma_high = sigma_mid
        
        return sigma_mid


class ScenarioSimulator:
    """Simulateur de scénarios pour la couverture"""
    
    def __init__(self, S0, K, T, r, sigma, quantity_kg):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.quantity_kg = quantity_kg
        self.engine = BlackScholesEngine(S0, K, T, r, sigma, 'call')
        self.option_price = self.engine.price()
        self.premium_total = self.option_price * quantity_kg
    
    def simulate_scenario(self, final_price):
        """Simule un scénario de prix final"""
        # Coût sans couverture
        cost_without_hedge = final_price * self.quantity_kg
        
        # Gain de l'option
        option_payoff = max(final_price - self.K, 0) * self.quantity_kg
        
        # Coût avec couverture
        cost_with_hedge = cost_without_hedge - option_payoff + self.premium_total
        
        # Économie réalisée
        saving = cost_without_hedge - cost_with_hedge
        
        # Prix unitaire effectif
        effective_unit_price = cost_with_hedge / self.quantity_kg
        
        return {
            'final_price': final_price,
            'cost_without_hedge': cost_without_hedge,
            'option_payoff': option_payoff,
            'premium_paid': self.premium_total,
            'cost_with_hedge': cost_with_hedge,
            'saving': saving,
            'effective_unit_price': effective_unit_price
        }
    
    def generate_scenarios(self):
        """Génère les 3 scénarios standard + matrice complète"""
        scenarios = {
            'hausse': self.simulate_scenario(self.S0 * 1.4),
            'baisse': self.simulate_scenario(self.S0 * 0.8),
            'constante': self.simulate_scenario(self.S0)
        }
        
        # Matrice de sensibilité
        prices = np.linspace(self.S0 * 0.5, self.S0 * 1.8, 20)
        sensitivity_matrix = [self.simulate_scenario(p) for p in prices]
        
        return scenarios, sensitivity_matrix, prices


# ============================================================================
# INTERFACE UTILISATEUR
# ============================================================================

def main():
    # En-tête principal
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.markdown("""
        <div style="text-align: center; margin-bottom: 2rem;">
            <span style="font-size: 3rem;">🍫</span>
            <h1 style="margin-bottom: 0;">CHOCOLATERIE KENGNE</h1>
            <p style="color: #8892b0; letter-spacing: 2px;">RISK MANAGEMENT SYSTEM</p>
            <div style="height: 2px; background: linear-gradient(90deg, transparent, #2ecc71, transparent); width: 60%; margin: 0 auto;"></div>
        </div>
        """, unsafe_allow_html=True)
    
    # Sidebar - Paramètres d'entrée
    with st.sidebar:
        st.markdown("""
        <div style="text-align: center; margin-bottom: 1.5rem;">
            <span style="font-size: 2rem;">⚙️</span>
            <h3 style="margin: 0;">PARAMÈTRES</h3>
            <p style="color: #5a6a8a; font-size: 0.8rem;">Black-Scholes Inputs</p>
        </div>
        """, unsafe_allow_html=True)
        
        # Type d'option
        option_type = st.selectbox(
            "TYPE DE CONTRAT",
            ["Call (Achat)", "Put (Vente)"],
            help="Call : droit d'acheter | Put : droit de vendre"
        )
        option_type_code = 'call' if 'Call' in option_type else 'put'
        
        st.markdown("---")
        
        # Paramètres marché
        S0 = st.number_input("💰 Prix spot (S₀)", min_value=500.0, max_value=10000.0, value=2500.0, step=50.0, help="Prix actuel du cacao (FCFA/kg)")
        K = st.number_input("🎯 Prix d'exercice (K)", min_value=500.0, max_value=10000.0, value=2600.0, step=50.0, help="Prix garanti par l'option")
        
        colT, colR = st.columns(2)
        with colT:
            T_days = st.number_input("⏱️ Échéance (jours)", min_value=1, max_value=365, value=90, step=5)
            T = T_days / 365
        with colR:
            r = st.number_input("🏦 Taux sans risque (%)", min_value=0.0, max_value=20.0, value=5.0, step=0.5) / 100
        
        sigma = st.number_input("📊 Volatilité σ (%)", min_value=5.0, max_value=100.0, value=35.0, step=1.0) / 100
        
        st.markdown("---")
        
        # Volume de couverture
        quantity_tonnes = st.number_input("📦 Volume à couvrir (tonnes)", min_value=1, max_value=1000, value=150, step=10)
        quantity_kg = quantity_tonnes * 1000
        
        st.markdown("---")
        
        # Calcul de la prime
        engine = BlackScholesEngine(S0, K, T, r, sigma, option_type_code)
        option_price = engine.price()
        
        st.markdown(f"""
        <div style="background: rgba(46, 204, 113, 0.1); border-radius: 12px; padding: 1rem; text-align: center; border: 1px solid rgba(46, 204, 113, 0.2);">
            <span style="font-size: 0.75rem; color: #8892b0;">PRIX UNITAIRE</span><br>
            <span style="font-size: 1.8rem; font-weight: 800; color: #2ecc71;">{option_price:,.2f}</span>
            <span style="font-size: 0.8rem;">FCFA/kg</span><br>
            <span style="font-size: 0.7rem; color: #5a6a8a;">Prime totale: {option_price * quantity_kg:,.0f} FCFA</span>
        </div>
        """, unsafe_allow_html=True)
        
        # Bouton de simulation
        st.markdown("<br>", unsafe_allow_html=True)
        run_simulation = st.button("🚀 LANCER LA SIMULATION", use_container_width=True)
    
    # ========================================================================
    # ZONE PRINCIPALE - RÉSULTATS
    # ========================================================================
    
    if run_simulation:
        # Animation de chargement
        with st.spinner("🔮 Calcul des probabilités et simulation des scénarios..."):
            time.sleep(0.5)
        
        # Recalcul avec les paramètres actuels
        engine = BlackScholesEngine(S0, K, T, r, sigma, option_type_code)
        option_price = engine.price()
        greeks = engine.get_all_greeks()
        
        # Simulation des scénarios
        simulator = ScenarioSimulator(S0, K, T, r, sigma, quantity_kg)
        scenarios, sensitivity_matrix, price_range = simulator.generate_scenarios()
        
        # ====================================================================
        # LIGNE 1 : METRIQUES CLÉS (KPI)
        # ====================================================================
        
        st.markdown("### 📈 POSITION DE COUVERTURE")
        
        col1, col2, col3, col4 = st.columns(4)
        
        with col1:
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-label">PRIME TOTALE</div>
                <div class="metric-value">{option_price * quantity_kg:,.0f}</div>
                <div class="metric-sub">FCFA</div>
            </div>
            """, unsafe_allow_html=True)
        
        with col2:
            color = "#2ecc71" if engine.delta() > 0 else "#e74c3c"
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-label">DELTA DE COUVERTURE</div>
                <div class="metric-value" style="color: {color};">{engine.delta():.4f}</div>
                <div class="metric-sub">{engine.delta() * quantity_kg:,.0f} kg à acheter</div>
            </div>
            """, unsafe_allow_html=True)
        
        with col3:
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-label">POINT MORT (BREAK-EVEN)</div>
                <div class="metric-value">{option_price + K:,.0f}</div>
                <div class="metric-sub">FCFA/kg</div>
            </div>
            """, unsafe_allow_html=True)
        
        with col4:
            protection_level = "100%" if engine.delta() > 0.5 else f"{engine.delta()*100:.1f}%"
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-label">NIVEAU DE PROTECTION</div>
                <div class="metric-value">{protection_level}</div>
                <div class="metric-sub">Contre la hausse</div>
            </div>
            """, unsafe_allow_html=True)
        
        # ====================================================================
        # LIGNE 2 : GREECS (ANALYSE DE SENSIBILITÉ)
        # ====================================================================
        
        st.markdown("### 🎛️ ANALYSE DES SENSIBILITÉS (GREEKS)")
        st.markdown("<p style='color: #8892b0; font-size: 0.8rem; margin-top: -1rem;'>Mesures de risque pour une gestion dynamique</p>", unsafe_allow_html=True)
        
        col1, col2, col3, col4, col5 = st.columns(5)
        
        greeks_labels = {
            'Delta (Δ)': ('Sensibilité au prix spot', '📈', '#2ecc71'),
            'Gamma (Γ)': ('Accélération du Delta', '⚡', '#f1c40f'),
            'Theta (Θ)': ('Perte temporelle par jour', '⏰', '#e74c3c'),
            'Vega (ν)': ('Sensibilité à la volatilité', '🌊', '#3498db'),
            'Rho (ρ)': ('Sensibilité au taux', '🏦', '#9b59b6')
        }
        
        cols = [col1, col2, col3, col4, col5]
        for idx, (name, value) in enumerate(greeks.items()):
            label, icon, color = greeks_labels[name]
            formatted_value = f"{value:+,.4f}" if name == 'Theta (Θ)' else f"{value:+.4f}"
            with cols[idx]:
                st.markdown(f"""
                <div style="background: rgba(18, 24, 36, 0.8); border-radius: 12px; padding: 1rem; text-align: center; border-left: 3px solid {color};">
                    <span style="font-size: 1.2rem;">{icon}</span>
                    <div style="font-size: 0.7rem; color: #8892b0;">{label}</div>
                    <div style="font-size: 1.1rem; font-weight: 700; color: {color};">{formatted_value}</div>
                    <div style="font-size: 0.65rem; color: #5a6a8a;">{name}</div>
                </div>
                """, unsafe_allow_html=True)
        
        # ====================================================================
        # LIGNE 3 : SCÉNARIOS PRINCIPAUX (STYLE ACTION)
        # ====================================================================
        
        st.markdown("### 🎬 SIMULATION DES SCÉNARIOS")
        st.markdown("<p style='color: #8892b0; font-size: 0.8rem; margin-top: -1rem;'>Analyse dynamique des 3 cas de figure</p>", unsafe_allow_html=True)
        
        # Cartes pour les 3 scénarios
        cols = st.columns(3)
        
        scenario_config = {
            'hausse': {'icon': '📈🔥', 'title': 'HAUSSE FORTE', 'color': '#e74c3c', 'badge': 'danger', 'mult': 1.4},
            'baisse': {'icon': '📉❄️', 'title': 'EFFONDREMENT', 'color': '#3498db', 'badge': 'warning', 'mult': 0.8},
            'constante': {'icon': '➡️⚖️', 'title': 'STABLE', 'color': '#f1c40f', 'badge': 'warning', 'mult': 1.0}
        }
        
        for idx, (key, config) in enumerate(scenario_config.items()):
            scenario = scenarios[key]
            final_price = scenario['final_price']
            
            with cols[idx]:
                gain_color = "#2ecc71" if scenario['saving'] > 0 else "#e74c3c"
                gain_sign = "+" if scenario['saving'] > 0 else ""
                
                st.markdown(f"""
                <div style="background: linear-gradient(135deg, rgba(18,24,36,0.95) 0%, rgba(13,18,32,0.95) 100%); border-radius: 16px; padding: 1rem; border: 1px solid rgba({int(config['color'][1:3],16)},{int(config['color'][3:5],16)},{int(config['color'][5:7],16)},0.3);">
                    <div style="text-align: center;">
                        <span style="font-size: 2rem;">{config['icon']}</span>
                        <h4 style="margin: 0; color: {config['color']};">{config['title']}</h4>
                        <span class="badge-{config['badge']}" style="display: inline-block; margin-top: 0.3rem;">{final_price:,.0f} FCFA/kg</span>
                    </div>
                    <div style="height: 1px; background: rgba(255,255,255,0.1); margin: 0.8rem 0;"></div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #5a6a8a;">Sans couverture:</span>
                        <span>{scenario['cost_without_hedge']/1e6:,.1f} M FCFA</span>
                    </div>
                    <div style="display: flex; justify-content: space-between; margin-top: 0.3rem;">
                        <span style="color: #5a6a8a;">Avec couverture:</span>
                        <span style="color: #2ecc71; font-weight: 600;">{scenario['cost_with_hedge']/1e6:,.1f} M FCFA</span>
                    </div>
                    <div style="display: flex; justify-content: space-between; margin-top: 0.3rem;">
                        <span style="color: #5a6a8a;">Gain/Perte:</span>
                        <span style="color: {gain_color}; font-weight: 700;">{gain_sign}{scenario['saving']/1e6:,.1f} M FCFA</span>
                    </div>
                    <div style="height: 1px; background: rgba(255,255,255,0.1); margin: 0.8rem 0;"></div>
                    <div style="text-align: center;">
                        <span style="font-size: 0.7rem; color: #5a6a8a;">Prix unitaire effectif</span><br>
                        <span style="font-size: 1.2rem; font-weight: 700; color: {config['color']};">{scenario['effective_unit_price']:,.0f}</span>
                        <span style="font-size: 0.7rem;"> FCFA/kg</span>
                    </div>
                </div>
                """, unsafe_allow_html=True)
        
        # ====================================================================
        # LIGNE 4 : GRAPHIQUES INTERACTIFS
        # ====================================================================
        
        st.markdown("### 📊 ANALYSE VISUELLE")
        
        col_chart1, col_chart2 = st.columns(2)
        
        with col_chart1:
            # Graphique des payoffs
            fig_payoff = go.Figure()
            
            # Payoff de l'option
            strikes = np.linspace(S0 * 0.5, S0 * 1.8, 100)
            if option_type_code == 'call':
                payoffs = np.maximum(strikes - K, 0)
                payoff_label = "Payoff du Call"
            else:
                payoffs = np.maximum(K - strikes, 0)
                payoff_label = "Payoff du Put"
            
            fig_payoff.add_trace(go.Scatter(
                x=strikes, y=payoffs,
                mode='lines',
                name=payoff_label,
                line=dict(color='#2ecc71', width=3),
                fill='tozeroy',
                fillcolor='rgba(46, 204, 113, 0.2)'
            ))
            
            # Ligne du strike
            fig_payoff.add_vline(x=K, line_dash="dash", line_color="#f1c40f", annotation_text=f"K = {K:,.0f}")
            
            # Ligne du prix spot
            fig_payoff.add_vline(x=S0, line_dash="dot", line_color="#3498db", annotation_text=f"S₀ = {S0:,.0f}")
            
            fig_payoff.update_layout(
                title="Payoff de l'option à l'échéance",
                xaxis_title="Prix du cacao (FCFA/kg)",
                yaxis_title="Gain (FCFA/kg)",
                template="plotly_dark",
                paper_bgcolor="rgba(0,0,0,0)",
                plot_bgcolor="rgba(18,24,36,0.8)",
                font=dict(color="#e0e0e0"),
                height=400
            )
            
            st.plotly_chart(fig_payoff, use_container_width=True)
        
        with col_chart2:
            # Graphique coût avec/sans couverture
            fig_cost = go.Figure()
            
            prices = [s['final_price'] for s in sensitivity_matrix]
            without_hedge = [s['cost_without_hedge']/1e6 for s in sensitivity_matrix]
            with_hedge = [s['cost_with_hedge']/1e6 for s in sensitivity_matrix]
            
            fig_cost.add_trace(go.Scatter(
                x=prices, y=without_hedge,
                mode='lines',
                name='Sans couverture',
                line=dict(color='#e74c3c', width=2, dash='dot')
            ))
            
            fig_cost.add_trace(go.Scatter(
                x=prices, y=with_hedge,
                mode='lines',
                name='Avec couverture',
                line=dict(color='#2ecc71', width=3),
                fill='tozeroy',
                fillcolor='rgba(46, 204, 113, 0.1)'
            ))
            
            # Point seuil
            break_even_price = option_price + K
            fig_cost.add_vline(x=break_even_price, line_dash="dash", line_color="#f1c40f", 
                               annotation_text=f"Seuil: {break_even_price:,.0f}")
            
            fig_cost.update_layout(
                title="Coût total d'approvisionnement (Millions FCFA)",
                xaxis_title="Prix final du cacao (FCFA/kg)",
                yaxis_title="Coût total (Millions FCFA)",
                template="plotly_dark",
                paper_bgcolor="rgba(0,0,0,0)",
                plot_bgcolor="rgba(18,24,36,0.8)",
                font=dict(color="#e0e0e0"),
                height=400
            )
            
            st.plotly_chart(fig_cost, use_container_width=True)
        
        # ====================================================================
        # LIGNE 5 : TABLEAU DE BORD DÉTAILLÉ
        # ====================================================================
        
        with st.expander("📋 TABLEAU DÉTAILLÉ DES SCÉNARIOS", expanded=False):
            # Création du DataFrame
            scenario_data = []
            for key, config in scenario_config.items():
                s = scenarios[key]
                scenario_data.append({
                    'Scénario': config['title'],
                    'Prix final (FCFA/kg)': f"{s['final_price']:,.0f}",
                    'Variation': f"{((s['final_price']/S0)-1)*100:+.1f}%",
                    'Sans couverture (M FCFA)': f"{s['cost_without_hedge']/1e6:,.1f}",
                    'Gain option (M FCFA)': f"{s['option_payoff']/1e6:,.1f}",
                    'Prime (M FCFA)': f"{s['premium_paid']/1e6:,.1f}",
                    'Avec couverture (M FCFA)': f"{s['cost_with_hedge']/1e6:,.1f}",
                    'Économie (M FCFA)': f"{s['saving']/1e6:+,.1f}",
                    'Prix unitaire effectif': f"{s['effective_unit_price']:,.0f}"
                })
            
            df = pd.DataFrame(scenario_data)
            st.dataframe(df, use_container_width=True, hide_index=True)
        
        # ====================================================================
        # INTERPRÉTATION AUTOMATIQUE POUR LE DG
        # ====================================================================
        
        st.markdown("### 🎯 RECOMMANDATION EXÉCUTIVE")
        
        # Analyse automatique
        worst_case = scenarios['hausse']['cost_without_hedge'] - scenarios['hausse']['cost_with_hedge']
        best_case = scenarios['baisse']['saving']
        
        if worst_case > 0:
            recommendation = "✅ **COUVERTURE RECOMMANDÉE** : La protection est rentable dans le scénario de hausse."
            recommendation_color = "#2ecc71"
        else:
            recommendation = "⚠️ **ANALYSE APPROFONDIE NÉCESSAIRE** : La prime de l'option est élevée."
            recommendation_color = "#f1c40f"
        
        st.markdown(f"""
        <div style="background: linear-gradient(135deg, rgba(46,204,113,0.1) 0%, rgba(46,204,113,0.02) 100%); border-radius: 16px; padding: 1.2rem; border: 1px solid rgba(46,204,113,0.2);">
            <div style="display: flex; align-items: center; gap: 1rem;">
                <span style="font-size: 2rem;">💼</span>
                <div>
                    <div style="font-size: 0.8rem; color: #8892b0;">POUR LE DIRECTEUR GÉNÉRAL</div>
                    <div style="font-size: 1rem; color: {recommendation_color};">{recommendation}</div>
                    <div style="font-size: 0.85rem; color: #5a6a8a; margin-top: 0.5rem;">
                        • Protection maximale à {scenarios['hausse']['effective_unit_price']:,.0f} FCFA/kg<br>
                        • Prime totale: {option_price * quantity_kg/1e6:.1f} M FCFA<br>
                        • Économie potentielle en cas de hausse: {worst_case/1e6:+.1f} M FCFA
                    </div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)
        
    else:
        # État initial - message d'accueil
        st.markdown("""
        <div style="text-align: center; padding: 3rem;">
            <span style="font-size: 4rem;">🍫📊</span>
            <h2>Système de Gestion des Risques</h2>
            <p style="color: #8892b0; font-size: 1.1rem;">
                Configurez les paramètres dans le panneau latéral<br>
                et cliquez sur <strong>"LANCER LA SIMULATION"</strong>
            </p>
            <div style="height: 2px; background: linear-gradient(90deg, transparent, #2ecc71, transparent); width: 40%; margin: 1rem auto;"></div>
            <p style="color: #5a6a8a; font-size: 0.85rem;">
                ⚡ Pricing Black-Scholes | Grecs | Simulation de scénarios | Couverture Delta
            </p>
        </div>
        """, unsafe_allow_html=True)
    
    # Footer
    st.markdown("""
    <div class="footer">
        <span>🍫 Chocolaterie Kengne — Risk Management System v1.0</span><br>
        <span style="font-size: 0.7rem;">Modèle Black-Scholes | Simulation dynamique | Analyse des Grecs</span>
    </div>
    """, unsafe_allow_html=True)


if __name__ == "__main__":
    main()

Overwriting Chocolaterie_Kengne_app.py


In [ ]:
!streamlit run Chocolaterie_Kengne_app.py